# Lecture 5 — Distribution Charts: Airbnb London

**Tasks:** price distributions by room type and listing activity by London borough.  
**Data source:** [Inside Airbnb — Get the Data](https://insideairbnb.com/get-the-data/) (London, England, United Kingdom).  
**Repository path:** `week05/lecture05_exercise.ipynb`

### Method notes
- Prices are capped by **filtering out listings above the 95th percentile** for the price-distribution chart; the cutoff is explicitly annotated.
- Room-type colours distinguish the two distributions; median lines use matching colours.
- Review activity excludes zero and missing `reviews_per_month` values. The two boroughs with the highest median activity are highlighted.
- The charts use Plotly and their outputs are saved as static PNGs in the notebook when this notebook is run, so the figures can also be viewed on GitHub.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from pathlib import Path

# Expected repository layout: data/airbnb_london.csv and week05/lecture05_exercise.ipynb
candidate_paths = [
    Path("../data/airbnb_london.csv"),
    Path("data/airbnb_london.csv"),
    Path("airbnb_london.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find airbnb_london.csv. Download the London summary listings CSV "
        "from https://insideairbnb.com/get-the-data/ and save it as data/airbnb_london.csv "
        "in the repository."
    )

df = pd.read_csv(data_path)
print(f"Loaded: {len(df):,} listings from {data_path}")
print(df.describe(include="all").transpose().head(20))

## Task 1 — Overlapping histogram: price by room type

Compare **Entire home/apt** and **Private room** listings on the same histogram. Shared rooms are excluded as requested. The 95th-percentile price cutoff is shown in the annotation, and each distribution has its own coloured median reference line.

In [ ]:
# Clean price and room type fields
required = {"price", "room_type"}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"Dataset is missing required columns: {sorted(missing)}")

price_df = df.copy()
price_df["price"] = (
    price_df["price"].astype(str)
    .str.replace(r"[$£€,]", "", regex=True)
    .str.strip()
)
price_df["price"] = pd.to_numeric(price_df["price"], errors="coerce")
price_df = price_df[(price_df["price"] > 0) & price_df["room_type"].notna()]

p95 = price_df["price"].quantile(0.95)
df_cap = price_df[price_df["price"] <= p95].copy()
room_order = ["Entire home/apt", "Private room"]
hist_df = df_cap[df_cap["room_type"].isin(room_order)].copy()

if hist_df.empty:
    raise ValueError("No usable price rows found for Entire home/apt and Private room.")

medians = hist_df.groupby("room_type")["price"].median()
counts = hist_df["room_type"].value_counts()
entire_median = medians.get("Entire home/apt", np.nan)
private_median = medians.get("Private room", np.nan)

if pd.notna(entire_median) and pd.notna(private_median):
    if entire_median > private_median:
        insight = "Entire homes have the higher median price; both distributions are right-skewed"
    elif private_median > entire_median:
        insight = "Private rooms have the higher median price; inspect the overlapping right-skewed tails"
    else:
        insight = "Both room types share the same median price, despite different price distributions"
else:
    insight = "Price distributions are right-skewed, with room-type medians shown below"

fig1 = px.histogram(
    hist_df,
    x="price",
    color="room_type",
    nbins=45,
    barmode="overlay",
    opacity=0.6,
    category_orders={"room_type": room_order},
    color_discrete_map={
        "Entire home/apt": "#2563EB",
        "Private room": "#E07A1F",
    },
    labels={"price": "Nightly price (£)", "count": "Number of listings", "room_type": "Room type"},
    title=insight,
)
fig1.update_layout(
    template="plotly_white",
    title_x=0.02,
    xaxis_title="Nightly price (£), capped at the 95th percentile",
    yaxis_title="Number of listings",
    legend_title_text="Room type",
    bargap=0.04,
    annotations=[
        dict(
            x=0.99, y=0.98, xref="paper", yref="paper",
            text=f"Price cap: 95th percentile = £{p95:,.0f}<br>Listings above cutoff excluded",
            showarrow=False, xanchor="right", yanchor="top",
            bgcolor="white", bordercolor="#777", borderwidth=1
        )
    ],
)
line_styles = {
    "Entire home/apt": "#1D4ED8",
    "Private room": "#C2410C",
}
for room in room_order:
    median_value = medians.get(room)
    if pd.notna(median_value):
        fig1.add_vline(
            x=float(median_value),
            line_dash="dash",
            line_width=2,
            line_color=line_styles[room],
            annotation_text=f"{room} median £{median_value:,.0f}",
            annotation_position="top",
            annotation_font_color=line_styles[room],
        )
fig1.show()

print("Price cutoff and medians:")
print(f"95th-percentile cutoff: £{p95:,.2f}")
print(medians.round(2))
print("\nListings plotted by room type:")
print(counts)

## Task 2 — Horizontal box plot: review activity by borough

`reviews_per_month` is used as a proxy for listing activity. Listings with zero or missing monthly reviews are excluded. Boroughs are sorted by median activity, with the two highest-median boroughs highlighted. Individual outlier points remain visible.

In [ ]:
required = {"reviews_per_month"}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"Dataset is missing required columns: {sorted(missing)}")

# Prefer the standard London borough/neighbourhood field when available.
borough_candidates = ["neighbourhood_cleansed", "neighbourhood", "borough"]
borough_col = next((c for c in borough_candidates if c in df.columns), None)
if borough_col is None:
    raise KeyError(
        "Could not find a borough column. Expected one of: "
        + ", ".join(borough_candidates)
    )

activity = df[[borough_col, "reviews_per_month"]].copy()
activity["reviews_per_month"] = pd.to_numeric(activity["reviews_per_month"], errors="coerce")
activity = activity.dropna(subset=[borough_col, "reviews_per_month"])
activity = activity[activity["reviews_per_month"] > 0]
activity = activity.rename(columns={borough_col: "borough"})
activity["borough"] = activity["borough"].astype(str).str.strip()
activity = activity[activity["borough"] != ""]

if activity.empty:
    raise ValueError("No listings with positive reviews_per_month values were found.")

borough_medians = activity.groupby("borough")["reviews_per_month"].median().sort_values(ascending=False)
top_two = borough_medians.head(2).index.tolist()
# Sort descending so the most active borough appears at the top of a horizontal chart.
borough_order = borough_medians.index.tolist()
activity["activity_group"] = np.where(activity["borough"].isin(top_two), "Top two by median", "Other boroughs")

if len(top_two) >= 2:
    insight2 = f"Highest median review activity: {top_two[0]} and {top_two[1]}"
elif top_two:
    insight2 = f"Highest median review activity: {top_two[0]}"
else:
    insight2 = "Review activity varies across London boroughs"

fig2 = px.box(
    activity,
    x="reviews_per_month",
    y="borough",
    orientation="h",
    points="outliers",
    category_orders={"borough": borough_order},
    color="activity_group",
    color_discrete_map={
        "Top two by median": "#0F766E",
        "Other boroughs": "#B8C2CC",
    },
    labels={
        "reviews_per_month": "Reviews per month (activity proxy)",
        "borough": "London borough / neighbourhood",
        "activity_group": "Median activity",
    },
    title=insight2,
)
fig2.update_layout(
    template="plotly_white",
    title_x=0.02,
    yaxis={"categoryorder": "array", "categoryarray": borough_order[::-1]},
    xaxis_title="Reviews per month (zero-review listings excluded)",
    yaxis_title="Borough / neighbourhood",
    legend_title_text="",
    boxmode="group",
)
# Add a median reference line for each borough, with a precise annotation in the hover data.
fig2.update_traces(
    boxmean=True,
    hovertemplate="<b>%{y}</b><br>Reviews/month: %{x:.2f}<extra></extra>"
)
fig2.add_annotation(
    x=0.99, y=0.01, xref="paper", yref="paper",
    text="Highlighted: two boroughs with the highest median reviews/month",
    showarrow=False, xanchor="right", yanchor="bottom",
    bgcolor="white", bordercolor="#777", borderwidth=1
)
fig2.show()

print("Top two boroughs by median reviews per month:")
print(borough_medians.head(2).round(2))
print(f"Listings included after excluding zero/missing reviews per month: {len(activity):,}")

## Export charts for GitHub

Run this cell after both charts have rendered. It writes static PNG versions to the notebook's working directory. The notebook itself will display saved outputs after you save it with the outputs present. If Plotly image export is unavailable, install Kaleido with `pip install -U kaleido` and rerun this cell.

In [ ]:
from pathlib import Path

output_dir = Path("figures")
output_dir.mkdir(exist_ok=True)

try:
    fig1.write_image(output_dir / "lecture05_price_distribution.png", scale=2)
    fig2.write_image(output_dir / "lecture05_borough_activity.png", scale=2)
    print(f"Saved charts to: {output_dir.resolve()}")
except Exception as exc:
    print("Static PNG export was not completed.")
    print("If needed, install Kaleido with: pip install -U kaleido")
    print(f"Export detail: {exc}")